# 02 — Data audit and exploratory analysis

Before forecasting, inspect the product as a dataset: dimensions, coordinates, temporal continuity, land mask, ranges, seasonality, and regional heterogeneity.

## Data-product warning

OISST is not a raw observation at every grid cell. NOAA combines satellite and in-situ observations, applies bias adjustments, and uses optimum interpolation to create a spatially complete daily field.

Consequently, some measured smoothness and predictability may come from the **analysis product** as well as from ocean dynamics:

\[
\text{forecastability of OISST}=\text{ocean signal}+\text{observation/analysis structure}.
\]

The EDA should therefore record spatial smoothness, missing/land structure, and any temporal processing boundaries. Conclusions are about forecasting **OISST v2.1**, unless validated against another product.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oisst_fno.data import open_oisst, validate_daily_time_axis

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
files = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))
if not files:
    raise FileNotFoundError("Run notebook 01 first.")

ds = open_oisst(files[-1])
validate_daily_time_axis(ds)
sst = ds["sst"]
print(ds.sizes)
print(sst.time.min().item(), sst.time.max().item())

In [ ]:
summary = pd.Series(
    {
        "n_days": sst.sizes["time"],
        "n_lat": sst.sizes["lat"],
        "n_lon": sst.sizes["lon"],
        "missing_fraction": float(sst.isnull().mean()),
        "sst_min": float(sst.min(skipna=True)),
        "sst_mean": float(sst.mean(skipna=True)),
        "sst_max": float(sst.max(skipna=True)),
    }
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sst.isel(time=0).plot(ax=ax)
ax.set_title(f"SST field — {str(sst.time.values[0])[:10]}")
plt.show()

In [ ]:
regional_mean = sst.mean(("lat", "lon"), skipna=True).to_series()
fig, ax = plt.subplots(figsize=(12, 4))
regional_mean.plot(ax=ax)
ax.set_ylabel("SST (°C)")
ax.set_title("Northeast Atlantic regional mean SST")
plt.show()

In [ ]:
monthly = regional_mean.groupby(regional_mean.index.month).agg(["mean", "std", "count"])
monthly.index.name = "month"
monthly

### Audit questions

- Are there missing days?
- Is the land mask effectively static?
- Are any finite values physically implausible?
- Does the spatial variance change by season?
- Are there service/version boundaries inside the study period?

Do not proceed to training until unexpected discontinuities are understood.